In [ ]:
import datetime
from os import listdir
from os.path import isfile, join
import json
import pandas as pd

In [ ]:

inp='../input/monthly/dam/'
pref='dam'
overwrite=False  # set True to regenerate csv files that already exist

In [ ]:
onlyfiles = [f for f in listdir(inp) if isfile(join(inp, f))]
for f in onlyfiles:

    if (f.split('_')[0]==pref and f.split('.')[1]=='json'):
        if not overwrite and isfile(join(inp, f.split('.')[0]+'.csv')):
            continue
        print(f.split('.'))
        with open(join(inp, f), "r") as read_file:
            dat = pd.DataFrame(json.load(read_file))

            dat = dat[~dat[0].isin(['Date:Time', 'Min', 'Max', 'Average', 'Total'])].copy()
            dat.columns=['datehour','price','volume']
            dat['price']=dat['price'].astype(float)
            dat['volume']=dat['volume'].astype(float)

            # datehour is either "(2016/01/01) 00-01" or "(9/1/2026) 00-01"
            parts=dat.datehour.str.extract(r'\((?P<d>[^)]+)\)\s*(?P<h>\d+)-')
            dat['date']=pd.to_datetime(parts.d,format="%Y/%m/%d",errors='coerce').fillna(
                        pd.to_datetime(parts.d,format="%m/%d/%Y",errors='coerce'))
            dat['hour']=parts.h.astype(int)
            assert dat['date'].notna().all(), f

            dat=dat.drop(columns=['datehour'])
            dat.set_index(['date','hour'],inplace=True)
            dat.to_csv(inp+f.split('.')[0]+'.csv')